# 01 — Data Audit & Quality Check

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook validates the data collected by our pipeline and establishes a data quality baseline.

**Sections:**
1. Setup & Database Connection
2. Table Schemas & Row Counts
3. Predictions Table — Quality Checks
4. Incidents Table — Quality Checks
5. Stations Table — Validation
6. Collection Coverage Over Time
7. GTFS Schedule Data Audit
8. Data Dictionary Summary

## 1. Setup & Database Connection

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Paths
PROJECT_DIR = Path("..")
DB_PATH = PROJECT_DIR / "data" / "wmata.db"
GTFS_DIR = PROJECT_DIR / "data" / "gtfs_rail"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")
print(f"Database size: {DB_PATH.stat().st_size / 1024:.0f} KB")

## 2. Table Schemas & Row Counts

In [ ]:
# List all tables and their row counts
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

for table in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn).iloc[0, 0]
    print(f"\n{'='*60}")
    print(f"Table: {table} — {count:,} rows")
    print(f"{'='*60}")
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    print(schema[["name", "type", "notnull", "pk"]].to_string(index=False))

## 3. Predictions Table — Quality Checks

In [ ]:
# Load predictions
df_pred = pd.read_sql("SELECT * FROM predictions", conn)
df_pred["collected_at"] = pd.to_datetime(df_pred["collected_at"])
print(f"Shape: {df_pred.shape}")
print(f"Date range: {df_pred['collected_at'].min()} to {df_pred['collected_at'].max()}")
df_pred.head(10)

In [ ]:
# Null counts per column
null_counts = df_pred.isnull().sum()
null_pct = (df_pred.isnull().sum() / len(df_pred) * 100).round(2)
null_report = pd.DataFrame({"nulls": null_counts, "pct": null_pct})
print("Null values per column:")
print(null_report.to_string())

In [ ]:
# Analyze the 'minutes' column — key field for delay detection
print("Unique values in 'minutes' column:")
print(df_pred["minutes"].value_counts().sort_index().to_string())
print(f"\nTotal unique values: {df_pred['minutes'].nunique()}")

In [ ]:
# Categorize 'minutes' values
def categorize_minutes(val):
    if val in ("ARR", "BRD"):
        return val
    elif val in ("---", "", "--", None):
        return "No Data"
    else:
        try:
            int(val)
            return "Numeric"
        except (ValueError, TypeError):
            return f"Other: {val}"

df_pred["minutes_category"] = df_pred["minutes"].apply(categorize_minutes)
cat_counts = df_pred["minutes_category"].value_counts()
cat_pct = (cat_counts / len(df_pred) * 100).round(2)

print("Minutes field categories:")
print(pd.DataFrame({"count": cat_counts, "pct": cat_pct}).to_string())

In [ ]:
# Distribution of numeric minutes values
numeric_mask = df_pred["minutes_category"] == "Numeric"
df_numeric = df_pred.loc[numeric_mask].copy()
df_numeric["minutes_int"] = df_numeric["minutes"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_numeric["minutes_int"], bins=range(0, df_numeric["minutes_int"].max() + 2),
             edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Minutes to Arrival")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Predicted Minutes to Arrival")

# By line
line_colors = {"RD": "red", "BL": "blue", "OR": "orange", "SV": "silver",
               "GR": "green", "YL": "gold"}
for line, color in line_colors.items():
    subset = df_numeric[df_numeric["line"] == line]["minutes_int"]
    if len(subset) > 0:
        axes[1].hist(subset, bins=range(0, 30), alpha=0.5, label=line, color=color)
axes[1].set_xlabel("Minutes to Arrival")
axes[1].set_ylabel("Count")
axes[1].set_title("Minutes to Arrival by Line")
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports" / "figures" / "minutes_distribution.png", dpi=150)
plt.show()

print(f"\nNumeric minutes summary statistics:")
print(df_numeric["minutes_int"].describe())

In [ ]:
# Line distribution
print("Records per line:")
line_counts = df_pred["line"].value_counts()
print(line_counts.to_string())

# Check for unexpected line values
expected_lines = {"RD", "BL", "OR", "SV", "GR", "YL"}
unexpected = set(df_pred["line"].unique()) - expected_lines
if unexpected:
    print(f"\nWARNING: Unexpected line values found: {unexpected}")
    for val in unexpected:
        print(f"  '{val}': {(df_pred['line'] == val).sum()} records")
else:
    print("\nAll line values are expected metro lines.")

In [ ]:
# Station coverage — are all 102 stations represented?
stations_in_predictions = df_pred["location_code"].nunique()
stations_in_table = pd.read_sql("SELECT COUNT(*) FROM stations", conn).iloc[0, 0]

print(f"Stations in predictions: {stations_in_predictions}")
print(f"Stations in stations table: {stations_in_table}")

# Which stations are missing from predictions?
all_stations = set(pd.read_sql("SELECT station_code FROM stations", conn)["station_code"])
seen_stations = set(df_pred["location_code"].unique())
missing = all_stations - seen_stations
if missing:
    missing_names = pd.read_sql(
        f"SELECT station_code, station_name FROM stations WHERE station_code IN ({','.join('?' for _ in missing)})",
        conn, params=list(missing)
    )
    print(f"\nStations with NO predictions yet ({len(missing)}):")
    print(missing_names.to_string(index=False))
else:
    print("\nAll stations have prediction data.")

In [ ]:
# Duplicate check — same station, line, destination, minutes at same timestamp
dup_cols = ["collected_at", "line", "location_code", "destination_code", "group_num", "minutes"]
available_cols = [c for c in dup_cols if c in df_pred.columns]
duplicates = df_pred.duplicated(subset=available_cols, keep=False).sum()
print(f"Exact duplicate rows (same timestamp + key fields): {duplicates:,}")
print(f"Percentage: {duplicates / len(df_pred) * 100:.2f}%")

## 4. Incidents Table — Quality Checks

In [ ]:
# Load incidents
df_inc = pd.read_sql("SELECT * FROM incidents", conn)
df_inc["collected_at"] = pd.to_datetime(df_inc["collected_at"])
print(f"Shape: {df_inc.shape}")
print(f"Date range: {df_inc['collected_at'].min()} to {df_inc['collected_at'].max()}")
print(f"\nNull counts:")
print(df_inc.isnull().sum().to_string())
df_inc.head()

In [ ]:
# Incident types and affected lines
print("Incident types:")
print(df_inc["incident_type"].value_counts().to_string())

print(f"\nUnique incident IDs: {df_inc['incident_id'].nunique()}")
print(f"Total incident rows: {len(df_inc)} (same incident re-collected each cycle)")

print("\nLines affected (raw values):")
print(df_inc["lines_affected"].value_counts().to_string())

In [ ]:
# Incident deduplication analysis
# Same incident_id appears every 5-min collection cycle
if df_inc["incident_id"].notna().any():
    inc_repeats = df_inc.groupby("incident_id").agg(
        times_collected=("collected_at", "count"),
        first_seen=("collected_at", "min"),
        last_seen=("collected_at", "max"),
        incident_type=("incident_type", "first"),
        lines=("lines_affected", "first")
    ).sort_values("times_collected", ascending=False)
    print("Unique incidents with collection frequency:")
    print(inc_repeats.to_string())
else:
    print("No incident IDs available yet.")

## 5. Stations Table — Validation

In [ ]:
# Load stations
df_stations = pd.read_sql("SELECT * FROM stations", conn)
print(f"Total stations: {len(df_stations)}")
print(f"\nNull counts:")
print(df_stations.isnull().sum().to_string())

# Stations per line
for i in range(1, 5):
    col = f"line_code{i}"
    counts = df_stations[col].dropna().value_counts()
    if len(counts) > 0:
        print(f"\n{col}: {counts.to_dict()}")

# Transfer stations (multi-line)
multi_line = df_stations[df_stations["line_code2"].notna()]
print(f"\nTransfer stations (serve 2+ lines): {len(multi_line)}")
print(multi_line[["station_code", "station_name", "line_code1", "line_code2"]].to_string(index=False))

In [ ]:
# Map stations geographically
fig, ax = plt.subplots(figsize=(10, 10))

for _, row in df_stations.iterrows():
    color = line_colors.get(row["line_code1"], "gray")
    ax.scatter(row["lon"], row["lat"], c=color, s=30, zorder=5)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("WMATA Station Locations (colored by primary line)")

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, label=l, markersize=8)
                   for l, c in line_colors.items()]
ax.legend(handles=legend_elements, loc="upper left")

plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports" / "figures" / "station_map.png", dpi=150)
plt.show()

## 6. Collection Coverage Over Time

In [ ]:
# Records collected per collection cycle (grouped by collected_at)
cycle_counts = df_pred.groupby("collected_at").size().reset_index(name="records")
cycle_counts["collected_at"] = pd.to_datetime(cycle_counts["collected_at"])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Records per cycle
axes[0].plot(cycle_counts["collected_at"], cycle_counts["records"], ".-", markersize=3)
axes[0].set_ylabel("Records per Cycle")
axes[0].set_title("Data Collection: Records per 2-Minute Cycle")
axes[0].axhline(y=cycle_counts["records"].median(), color="red", linestyle="--",
                label=f"Median: {cycle_counts['records'].median():.0f}")
axes[0].legend()

# Cumulative records
cycle_counts["cumulative"] = cycle_counts["records"].cumsum()
axes[1].plot(cycle_counts["collected_at"], cycle_counts["cumulative"], color="green")
axes[1].set_ylabel("Cumulative Records")
axes[1].set_xlabel("Time (UTC)")
axes[1].set_title("Cumulative Predictions Collected")

plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports" / "figures" / "collection_coverage.png", dpi=150)
plt.show()

print(f"Total collection cycles: {len(cycle_counts)}")
print(f"Records per cycle — mean: {cycle_counts['records'].mean():.0f}, "
      f"min: {cycle_counts['records'].min()}, max: {cycle_counts['records'].max()}")

# Check for gaps (cycles more than 3 min apart)
cycle_counts["gap_seconds"] = cycle_counts["collected_at"].diff().dt.total_seconds()
gaps = cycle_counts[cycle_counts["gap_seconds"] > 180]
if len(gaps) > 0:
    print(f"\nWARNING: {len(gaps)} collection gap(s) > 3 minutes detected:")
    print(gaps[["collected_at", "gap_seconds"]].to_string(index=False))
else:
    print("\nNo collection gaps detected (all cycles within 3 min of each other).")

In [ ]:
# Hourly collection volume
df_pred["hour_utc"] = df_pred["collected_at"].dt.hour
hourly = df_pred.groupby("hour_utc").size()

fig, ax = plt.subplots(figsize=(12, 5))
hourly.plot(kind="bar", color="steelblue", edgecolor="black", ax=ax)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Total Records")
ax.set_title("Records Collected by Hour (UTC)")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "reports" / "figures" / "hourly_volume.png", dpi=150)
plt.show()

## 7. GTFS Schedule Data Audit

In [ ]:
# Load and inspect GTFS files
gtfs_files = {}
for f in GTFS_DIR.glob("*.txt"):
    gtfs_files[f.stem] = pd.read_csv(f)
    print(f"{f.name}: {len(gtfs_files[f.stem]):,} rows, {gtfs_files[f.stem].shape[1]} columns")

print(f"\nTotal GTFS files: {len(gtfs_files)}")

In [ ]:
# Inspect stop_times — scheduled arrival/departure times per trip per stop
stop_times = gtfs_files["stop_times"]
print("stop_times columns:", list(stop_times.columns))
print(f"\nUnique trips: {stop_times['trip_id'].nunique():,}")
print(f"Unique stops: {stop_times['stop_id'].nunique()}")
stop_times.head(10)

In [ ]:
# Inspect stops — station metadata from GTFS
stops = gtfs_files["stops"]
print("stops columns:", list(stops.columns))
print(f"\nTotal stops in GTFS: {len(stops)}")
print(f"\nSample stop IDs:")
print(stops[["stop_id", "stop_name"]].head(20).to_string(index=False))

In [ ]:
# Attempt to map GTFS stop_id to WMATA station_code
# GTFS uses platform-level stop IDs; WMATA API uses station codes (e.g., A01, B02)
print("GTFS stop_id samples:")
print(stops["stop_id"].head(20).tolist())
print(f"\nWMATA station_code samples:")
print(df_stations["station_code"].head(20).tolist())

# Check if GTFS stop_id contains WMATA station codes
wmata_codes = set(df_stations["station_code"])
gtfs_ids = set(stops["stop_id"])

direct_matches = wmata_codes & gtfs_ids
print(f"\nDirect matches (GTFS stop_id == WMATA station_code): {len(direct_matches)}")

# Try matching by name
if len(direct_matches) < 10:
    print("\nDirect ID match is low — attempting name-based matching...")
    # Normalize names for fuzzy matching
    stops["name_clean"] = stops["stop_name"].str.strip().str.lower()
    df_stations["name_clean"] = df_stations["station_name"].str.strip().str.lower()
    name_matches = pd.merge(df_stations[["station_code", "name_clean"]],
                            stops[["stop_id", "name_clean"]],
                            on="name_clean", how="inner")
    print(f"Name-based matches: {len(name_matches)}")
    if len(name_matches) > 0:
        print(name_matches.head(10).to_string(index=False))

In [ ]:
# Inspect trips and routes
trips = gtfs_files["trips"]
routes = gtfs_files["routes"]

print("Routes:")
print(routes[[c for c in routes.columns if c in
    ["route_id", "route_short_name", "route_long_name", "route_color"]]].to_string(index=False))

print(f"\nTrips per route:")
print(trips["route_id"].value_counts().to_string())

In [ ]:
# Calendar dates — which service days exist
if "calendar_dates" in gtfs_files:
    cal = gtfs_files["calendar_dates"]
    print("calendar_dates columns:", list(cal.columns))
    print(f"\nService IDs: {cal['service_id'].nunique()}")
    print(f"Date range: {cal['date'].min()} to {cal['date'].max()}")
    print(f"\nException types:")
    print(cal["exception_type"].value_counts().to_string())
elif "calendar" in gtfs_files:
    cal = gtfs_files["calendar"]
    print("calendar columns:", list(cal.columns))
    print(cal.head())
else:
    print("No calendar/calendar_dates file found.")

## 8. Data Dictionary Summary

In [ ]:
# Summary data dictionary
data_dict = pd.DataFrame([
    ["predictions.collected_at", "TEXT (ISO 8601 UTC)", "Timestamp when data was pulled from the API"],
    ["predictions.car", "TEXT", "Number of rail cars (6 or 8)"],
    ["predictions.destination", "TEXT", "Abbreviated destination name"],
    ["predictions.destination_code", "TEXT", "Station code of the destination (may be null)"],
    ["predictions.destination_name", "TEXT", "Full destination station name"],
    ["predictions.group_num", "TEXT", "Track group: 1 or 2 (indicates direction)"],
    ["predictions.line", "TEXT", "Metro line color: RD, BL, OR, SV, GR, YL (or --, No)"],
    ["predictions.location_code", "TEXT", "Station code where prediction was observed"],
    ["predictions.location_name", "TEXT", "Station name where prediction was observed"],
    ["predictions.minutes", "TEXT", "Minutes to arrival: numeric (1-30+), ARR, BRD, or ---"],
    ["", "", ""],
    ["incidents.incident_id", "TEXT", "Unique WMATA incident identifier"],
    ["incidents.incident_type", "TEXT", "Type: Delay or Alert"],
    ["incidents.description", "TEXT", "Full text description of the incident"],
    ["incidents.lines_affected", "TEXT", "Semicolon-separated line codes affected"],
    ["incidents.date_updated", "TEXT", "Last update time reported by WMATA"],
    ["", "", ""],
    ["stations.station_code", "TEXT", "Unique station identifier (e.g., A01, B02)"],
    ["stations.station_name", "TEXT", "Full station name"],
    ["stations.lat / lon", "REAL", "Geographic coordinates"],
    ["stations.line_code1-4", "TEXT", "Metro lines serving this station (up to 4)"],
    ["stations.together_station", "TEXT", "Connected station code (for transfers)"],
], columns=["Field", "Type", "Description"])

print(data_dict.to_string(index=False))

In [ ]:
# Final summary
print("=" * 60)
print("DATA AUDIT SUMMARY")
print("=" * 60)
print(f"Predictions:  {len(df_pred):>10,} rows")
print(f"Incidents:    {len(df_inc):>10,} rows")
print(f"Stations:     {len(df_stations):>10,} rows")
print(f"GTFS stops:   {len(stops):>10,} rows")
print(f"GTFS trips:   {len(trips):>10,} rows")
print(f"GTFS times:   {len(stop_times):>10,} rows")
print(f"\nCollection period: {df_pred['collected_at'].min()} to {df_pred['collected_at'].max()}")
print(f"Collection cycles: {len(cycle_counts)}")
print(f"Stations covered: {stations_in_predictions} / {stations_in_table}")
print(f"\nDelay threshold: 2 minutes (headway-based)")
print(f"Target: 100,000+ prediction rows")
print(f"Current progress: {len(df_pred)/100_000*100:.1f}%")
print("=" * 60)

conn.close()
print("\nDatabase connection closed.")